##Ingest constructors.json File
1.Read Data from circuits.csv using pyspark dataframe reader
2.Add Ingestion Metdata
    a.Add ingestion timestamp
    b.Add source
3.Write final dataframe to bronze schema

## Step1 .Read data 

In [0]:
%run ../00.common/01.environment_config

In [0]:
%run ../00.common/02.bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/constructors.json"
table_name = f"{catalog_name}.{bronze_schema}.constructors"

In [0]:
constructors_schema = 'constructorId STRING, name STRING, nationality STRING, url STRING'

In [0]:
# from pyspark.sql.types import StructType,StructField,StringType,FloatType

# circuits_schema = StructType(
#     [
#         StructField('circuitId', StringType(), True),
#         StructField('url', StringType(), True),
#         StructField('circuitName', StringType(), True),
#         StructField('lat', FloatType(), True),
#         StructField('long', FloatType(), True),
#         StructField('locality', StringType(), True),
#         StructField('country', StringType(), True)
#     ]
# )

In [0]:
constructors_df = (
    spark.read
    .format('json')
    .option('header',True)
    .schema(constructors_schema)
    .option('mode','FAILFAST')
    .load(source_file)
)

In [0]:
display(constructors_df)

## Step2 . Add Ingestion Metadata

In [0]:
from pyspark.sql import functions as F
constructors_final_df = add_ingestion_metadata(constructors_df)

##Step3 .Write Data to Delta Table

In [0]:
(
    constructors_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
-- SELECT * FROM formula1.bronze.circuits;

In [0]:
# display(spark.read.table(table_name))